# Manual Backpropagation

#### data

In [1]:
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch

In [2]:
words = open('../names.txt', 'r').read().splitlines()

In [3]:
letters = sorted(list(set(''.join(words))))
str_to_int = {s:i+1 for i, s in enumerate(letters)}
str_to_int['.'] = 0
int_to_str = {i:s for s, i in str_to_int.items()}
vocab_size = len(int_to_str)

In [ ]:
def build_dataset(words):
    context_length = 3
    X, Y = [], []

    for word in words:
        context = [0] * context_length
        for character in word + '.':
            index = str_to_int[character]
            Y.append(index)
            X.append(context)
            context = context[1:] + [index]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(1234)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtrain, Ytrain = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])

#### comapre gradients

In [5]:
def compare(s, dt,t):   
    exact = torch.all(dt == t.grad).item()
    approximate = torch.allclose(dt,  t.grad)
    max_diff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(exact):5s} | approximate: {str(approximate):5s} | maxdiff: {max_diff}')

#### initialisation

In [6]:
embedded_dim = 10
n_hidden = 200
context_length = 3

g = torch.Generator().manual_seed(6710)
C = torch.randn((vocab_size,embedded_dim),                  generator=g)

#make near 0 as if 0 can mask mistakes with gradient calcualtions
W1 = torch.randn((context_length * embedded_dim, n_hidden), generator=g) * (5/3) / (context_length * embedded_dim)**0.5
b1 = torch.randn(n_hidden,                                  generator=g) * 0.1

W2 = torch.randn((n_hidden, vocab_size),                    generator=g) * 0.1
b2 = torch.rand(27,                                         generator=g) * 0.1


bn_gain = torch.ones(1, n_hidden) * 0.1 + 1.0
bn_bias = torch.zeros(1, n_hidden) * 0.1

bn_mean_running = torch.zeros(1, n_hidden)
bn_std_running = torch.ones(1, n_hidden)

parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
for p in parameters:
    p.requires_grad=True

In [7]:
batch_size = 32

index = torch.randint(0, Xtrain.shape[0], (batch_size,), generator=g)
#Xb, Yb = Xtrain[index], Ytrain[index]

#### forward pass

In [8]:
embeddings = C[Xtrain[index]]
emb_concat = embeddings.view(embeddings.shape[0], -1)

#Linear layer 1
h_pre_bn = emb_concat @ W1 + b1

#Batch Normalisation Layer
bn_mean_current = 1/batch_size * h_pre_bn.sum(0, keepdim=True)
bn_diff = h_pre_bn - bn_mean_current
bn_diff_squared = bn_diff ** 2
bn_var = 1/(batch_size-1) * bn_diff_squared.sum(0, keepdim=True) #bessel's correction divide by n-1
bn_var_inv = (bn_var + 1e-5)**-0.5
h_norm = bn_diff * bn_var_inv #bnraw
bn_output = bn_gain * h_norm + bn_bias #hpreact

#Non-Linearity
h = torch.tanh(bn_output)

#Linear layer 2
logits = h @ W2 + b2

#Written out Cross Entropy Loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes #for numer
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(batch_size), Ytrain[index]].mean()

#Pytorch Backwards Pass (for comparison)
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, bn_output, h_norm,
          bn_var_inv, bn_var, bn_diff_squared, bn_diff, h_pre_bn, bn_mean_current,
          emb_concat, embeddings]:
    t.retain_grad()
loss.backward()
loss


tensor(3.5528, grad_fn=<NegBackward0>)

### backwards pass

In [9]:
#logprobs
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(batch_size), Ytrain[index]] = -1.0/batch_size
compare('logprobs', dlogprobs, logprobs)

#probs
dprobs = 1/probs * dlogprobs
compare('probs', dprobs, probs)

#counts_sum_inv
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
compare('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

#counts_sum
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
compare('counts_sum', dcounts_sum, counts_sum)

#counts (two paths: directly into probs, and through counts_sum)
dcounts = counts_sum_inv * dprobs
dcounts += torch.ones_like(counts) * dcounts_sum
compare('counts', dcounts, counts)

#norm_logits
dnorm_logits = norm_logits.exp() * dcounts
compare('norm_logits', dnorm_logits, norm_logits)

#logit_maxes
dlogit_maxes = (-1 * dnorm_logits).sum(1, keepdim=True)
compare('logit_maxes', dlogit_maxes, logit_maxes)

#logits (two paths: through norm_logits, and through logit_maxes)
dlogits = dnorm_logits * 1
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
compare('logits', dlogits, logits)

#Linear Layer 2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
compare('h', dh, h)
compare('W2', dW2, W2)
compare('b2', db2, b2)

#non-linearity layer
dbn_output = (1 - h**2) * dh
compare('bn_output', dbn_output, bn_output)

#Batch Normalisation Layer
dbn_gain = (h_norm * dbn_output).sum(0, keepdim=True)
dh_norm = bn_gain * dbn_output
dbn_bias = dbn_output.sum(0, keepdim=True)
compare('bn_gain', dbn_gain, bn_gain)
compare('h_norm', dh_norm, h_norm)
compare('bn_bias', dbn_bias, bn_bias)

dbn_diff = bn_var_inv * dh_norm
dbn_var_inv = (bn_diff * dh_norm).sum(0, keepdim=True)
dbn_var = (-0.5*(bn_var + 1e-5)**(-3/2)) * dbn_var_inv
compare('bn_var_inv', dbn_var_inv, bn_var_inv)
compare('bn_var', dbn_var, bn_var)

dbn_diff_squared = (1/(batch_size-1)) * torch.ones_like(bn_diff_squared) * dbn_var
dbn_diff += 2 * bn_diff * dbn_diff_squared
dh_pre_bn = dbn_diff.clone()
dbn_mean_current = (-torch.ones_like(bn_diff) * dbn_diff).sum(0)
dh_pre_bn += dbn_mean_current * 1/batch_size * torch.ones_like(h_pre_bn)
compare('bn_diff_squared', dbn_diff_squared, bn_diff_squared)
compare('bn_diff', dbn_diff, bn_diff)
compare('bn_mean_current', dbn_mean_current, bn_mean_current)
compare('h_pre_bn', dh_pre_bn, h_pre_bn)

#Linear layer 1 and embeddings
demb_concat = dh_pre_bn @ W1.T
dW1 = emb_concat.T @ dh_pre_bn
db1 = dh_pre_bn.sum(0)
dembeddings = demb_concat.view(embeddings.shape)
compare('emb_concat', demb_concat, emb_concat)
compare('W1', dW1, W1)
compare('b1', db1, b1)
compare('embeddings', dembeddings, embeddings)

#C (each embedding's gradient goes back to the row of the character that was looked up)
dC = torch.zeros_like(C)
for i in range(Xtrain[index].shape[0]):
    for j in range(Xtrain[index].shape[1]):
        ix = Xtrain[index][i,j]
        dC[ix] += dembeddings[i,j]
compare('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
bn_output       | exact: True  | approximate: True  | maxdiff: 0.0
bn_gain         | exact: True  | approximate: True  | maxdiff: 0.0
h_norm          | exact: True  | approximate: True  | maxdiff: 0.0
bn_bias         | exact: True  | approximate: True  | maxdiff:

#### simplifying back propagation for the Loss

In [10]:
dlogits = F.softmax(logits, 1)
dlogits[range(batch_size), Ytrain[index]] -= 1
dlogits /= batch_size

loss_quick = F.cross_entropy(logits, Ytrain[index])
print(loss_quick.item(), 'difference:', (loss_quick - loss).item())

compare('logits', dlogits, logits)

3.552813768386841 difference: 2.384185791015625e-07
logits          | exact: False | approximate: True  | maxdiff: 6.752088665962219e-09


#### Simplifying back propagation for the Batch Normalisation

In [11]:

bn_output_quick = bn_gain * (h_pre_bn - h_pre_bn.mean(0, keepdim=True)) / torch.sqrt(h_pre_bn.var(0, keepdim=True, unbiased=True) + 1e-5) + bn_bias
print('max difference', (bn_output_quick - bn_output).abs().max().item())


dh_pre_bn = bn_gain * bn_var_inv/batch_size * (batch_size*dbn_output - dbn_output.sum(0) - batch_size/(batch_size-1)*h_norm*(dbn_output * h_norm).sum(0))
compare('h_pre_bn', dh_pre_bn, h_pre_bn)

max difference 4.76837158203125e-07
h_pre_bn        | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10
